# 第70课 · 拆开能听懂 99 种语言的机器——Whisper 架构：log-Mel → Encoder-Decoder → 多任务头

**目标**：先读懂 Whisper 的四幕架构，再把它和 Aurora DSP / CTC 路线对照起来。

> **本课你只要能**：① 画四框图（log-Mel → Encoder → Decoder → token）  
> ② 说出与 CTC（L68–69）的一点不同（自回归 vs 对齐求和）。  
> 论文超参与消融默认当「进阶阅读」，主路径先瘦身。

🔗 **Aurora 连接**：80-dim log-Mel 即 L46–L47 的 `mel_spectrogram(n_mels=80)`（约 25 ms 窗 / 10 ms 帧移）。

← **上一课**　[L69 · CTC 前向算法](L69_ctc_forward.ipynb)

> 上节课学习了 **CTC 前向算法**：所有路径概率求和，O(T·S)（S=2L+1）纯 NumPy 实现。  
> 本课将探讨 **Whisper 架构解析**。

## 学习目标

学完本节，你应该能够：

1. 理解 Whisper 80-dim log-Mel 输入的生成过程及与 Aurora DSP 的参数对齐（`n_fft=400, hop=160`）
2. 描述编码器 Conv1d×2 + Transformer 的形状变换链（输入 3000→Conv 降采样→1500→Transformer→(1500, d_model)）
3. 解释交叉注意力中 Q 来自解码器、K/V 来自编码器，以及注意力权重（attention weight） shape `(B, T_dec, T_enc)`
4. 列出 Whisper 多任务头的四种输出及条件 token 前缀机制
5. 实现 `whisper_preprocess`，通过 30 s 静音、5 s 短音频补零、60 s 长音频截断三组断言

## Whisper / CTC 连接提示

Whisper 用的是交叉熵（cross-entropy）训练，而不是 CTC；L68–L69 则是理解“序列对齐基线”的前一站。
先把 CTC 的单调路径和 log 域 DP 看懂，再看 Whisper 就会知道它为什么选择 token-by-token 的自回归解码。

## 本课剧情：Whisper 为什么能“听懂 99 种语言”？

OpenAI 2022 年发布的 Whisper，在没有任何特殊工程的情况下，能转写 99 种语言的语音。秘密不是“魔法”，而是数据、结构和训练目标一起堆出来的。

**答案：超大规模弱监督 + Encoder-Decoder + 多任务头。**

Whisper 把整个 ASR 流水线压缩成一个架构：

```
音频 (16kHz)
   ↓ log-Mel (n_fft=400, hop=160, n_mels=80, 30s→3000帧)
   ↓ 2×Conv1D 降采样 → 1500步向量序列
   ↓ Transformer Encoder (多层 self-attention)
   ↓ 上下文向量 (B, 1500, d_model)
   ↓ Transformer Decoder (cross-attention到编码器)
   ↓ BPE token 概率 (B, T_dec, vocab_size)
   ↓ 文字
```

**三个关键设计**：

| 设计 | 目的 |
|---|---|
| 固定30秒窗口 (3000帧) | 统一输入形状，无需变长处理 |
| 2×Conv1D降采样 (stride=2) | 3000帧→1500步，减少 decoder 注意力计算量 |
| 多任务 token (`<en>`,`<transcribe>`,`<translate>`) | 同一模型完成转写/翻译/VAD，无需多个模型 |

**Aurora 连接**：`aurora.audio.mel.mel_spectrogram()` 与 Whisper 官方参数完全对齐（n_mels=80, n_fft=400, hop=160, sr=16000）——本节 `whisper_preprocess()` 直接调用它。

本节任务：实现 `whisper_preprocess(wav)` — 音频→`(1, 80, 3000)` Tensor。

### 追问：为什么恰好是 30 秒、恰好降到 1500 步？

上面的表格说"固定 30 秒窗口"是为了"统一输入形状"，"降采样到 1500 步"是为了"减少计算量"。但**为什么偏偏是 30 秒而不是 20 秒或 40 秒？为什么偏偏是 1500 而不是 1000 或 2000？**

先说结论：**这两个数字都不是靠某个公式"唯一推导"出来的，而是 OpenAI 做了大量实验后选定的工程折中（engineering trade-off）**——有点像相机厂商把照片分辨率定在某个值：再大一点画质提升有限但存储暴涨，再小一点又不够看清细节。下面把这个"折中"到底在权衡什么，用数字算给你看。

#### 折中点 1：自注意力的计算量是 T 的平方，不是 T 的一倍

Transformer 编码器里的自注意力，每一步都要跟**所有其他步**算一次相似度（第 3 节会细讲）。序列长度是 T 的话，要算的"两两相似度"总数是：

```
计算量 ∝ T × T = T²
```

这意味着：**序列长度减半，计算量不是减半，而是减到 1/4！** 这正好回答了老陈的追问——"降采样 2 倍对计算量有什么具体影响"：下面代码把 T=3000（不降采样）、1500（Whisper 实际选择）、750（再多降一次）的计算量都算了出来——1500 步只有 3000 步的 1/4 计算量，这就是"降一次"带来的实打实的收益。

#### 折中点 2：降采样越多，时间分辨率越粗，可能丢音素细节

每次 stride=2 降采样，相当于把"两帧合并成一步"。降采样后，每一步对应的时间跨度是：

```
1 步对应的时长 = hop(10ms) × 卷积总降采样倍数(2) = 20 ms
```

如果再多降采样一次（等效 4 倍），每一步就要覆盖 40 ms。问题在于：像"b"和"p"这种辅音的区别，往往就在几十毫秒内的爆破时刻——降采样太狠，这些短促的声学线索可能被"平均"进相邻步里，模型分辨起来更费劲。所以 Whisper 选 stride=2（20ms/步）是"计算量"和"时间分辨率"之间的折中点，而不是降采样越多越好。

#### 折中点 3：30 秒窗口要覆盖"一句话通常有多长"

音频窗口不是越长越好——窗口越长，折中点 1 里的 T² 计算量就越大。30 秒基本能装下大多数自然语言里一整句、甚至一小段话，同时不会让 T 大到离谱。如果你有一段 5 分钟的录音，Whisper 并不会一次性塞进模型——它会用**滑动的 30 秒窗口**（有重叠地）分段处理，再把结果拼接起来（这是 L71 之后会涉及的长音频转写策略）。所以"30 秒"更像是"每次咀嚼一口"的大小，而不是"一句话的硬性上限"。

**练习**：自己动手算一下，如果 Whisper 选择 20 秒窗口会怎样？下面的代码已经把 20/30/40 秒三种情况都算好了——自己读一遍输出，想想"省了多少计算量"和"能装下多长语音"之间的取舍。

In [ ]:
# 演示：窗口长度、降采样倍数如何影响序列长度 T 和自注意力计算量（T²）
def window_stats(seconds, sr=16000, hop=160, downsample=2):
    n_samples = seconds * sr
    n_frames = n_samples // hop
    T = n_frames // downsample
    compute = T ** 2
    return n_frames, T, compute

print("窗口长度对比（固定降采样2倍）：")
for seconds in [20, 30, 40]:
    n_frames, T, compute = window_stats(seconds)
    print(f"  {seconds:2d} s → mel帧数={n_frames:5d} → 降采样后 T={T:5d} → 自注意力计算量(T²)={compute:>10,}")

print("\n降采样倍数对比（固定30秒窗口）：")
baseline = None
for downsample in [1, 2, 4]:
    n_frames, T, compute = window_stats(30, downsample=downsample)
    baseline = compute if baseline is None else baseline
    print(f"  降采样{downsample}倍 → T={T:5d} → 计算量(T²)={compute:>10,}  (相对不降采样: {compute/baseline*100:.1f}%)")


In [ ]:
import torch
import numpy as np
from aurora.audio.mel import mel_spectrogram, power_to_db


## 1. Whisper 输入：80-dim log-Mel，shape `(80, 3000)`

### 从"采样率"到"帧"：为什么 hop=160？

音频的采样率是 16 kHz，意思是每秒记录 16000 个数字。30 秒音频就有 16000 × 30 = **480,000** 个采样点。

但我们不能把这 480,000 个点全部送给模型——太密集了。想象你看一部电影，如果每毫秒看一帧，一天就看完了。所以电影只给你每秒 24 帧。

同样的道理，我们可以这样"降抽样"音频：

```
每隔 160 个采样点提取一次特征 → 一"帧"（frame）

30 秒 = 480,000 个采样点
480,000 ÷ 160 = 3,000 帧
```

这个 160 叫 **hop（帧移）**或 **hop_length**。为什么是 160？
- 采样率 16 kHz，一秒 16,000 个点
- 160 个点的时长 = 160 / 16,000 = **0.01 秒 = 10 毫秒**
- 所以 hop=160 意味着每隔 10 ms 提一帧

你可以把这想象成"滑动窗口"：

```
采样点:  [S0, S1, S2, ..., S159, S160, ..., S319, S320, ...]
窗口1:  [S0  ~ S159]   → 第1帧特征
窗口2:          [S160 ~ S319]   → 第2帧特征（窗口向后移160个点）
窗口3:                  [S320 ~ S479]   → 第3帧特征
...
```

每个窗口内做短时傅立叶变换（STFT），提出频率信息，就得到了 mel 谱图。

**核心公式**：
```
帧数 = 总采样点数 / hop_length
     = 480,000 / 160
     = 3,000
```

---

Whisper 固定使用 **30 秒**音频窗口（不足补零，超出截断）。采样率 16 kHz，所以 30 s = 480 000 个采样点。STFT 参数：`n_fft=400`（25 ms 窗口）、`hop=160`（10 ms 帧移）。

**为什么是这些参数？**

- **n_fft=400**：采样率 16 kHz，400 个点 = 25 ms 的时间窗口。大约是人耳感知音素的时间尺度。
- **hop=160**：这是 n_fft 的 1/2.5，对应每 10 ms 采一帧——足够密集地追踪声调（pitch）和音素边界。

帧数为 `480000 // 160 = 3000`（官方约定），mel 维度固定 80。注意：Aurora 的 `mel_spectrogram` 默认 `center=True`（反射填充），实际返回 **3001** 帧（`480000 // 160 + 1`）——官方实现同样先多算一帧再裁掉末帧，练习步骤 3 的 truncate 正好抹平这一帧差。

---

Whisper 的 log-Mel 归一化（normalization）：

```
log_mel = log10(max(mel_power, 1e-10))
log_mel = max(log_mel, log_mel.max() - 8.0)
log_mel = (log_mel + 4.0) / 4.0
```

**这三行在干什么？**

**第一行**：`log10(max(mel_power, 1e-10))`
- 对频率能量取对数（log scale），压缩动态范围。
- `max(..., 1e-10)` 防止 log(0)，给极小值一个下界。

**第二行**：`max(log_mel, log_mel.max() - 8.0)`
- 找出 log_mel 的最大值，然后向下"剪裁"到 `最大值 - 8.0`。
- 作用：**削平离群值，限制动态范围**。
- 为什么是减 8.0？8.0 在 log10 尺度下是 10^8 倍的能量差，大约是人耳能区分的最大范围。
- 例如，如果 log_mel.max() = 0（指 10^0=1 的相对能量），那么 clip 下界是 -8.0（指 10^-8 ~ 极弱的声音）。任何低于 -8.0 的值（超级微弱的噪声）都被粗暴地设成 -8.0。

**第三行**：`(log_mel + 4.0) / 4.0`
- 线性变换，把值域缩放到 `[-1, 1]` 附近。
- 为什么是加 4.0 再除 4.0？
  - 第二行后，log_mel 的范围大约是 `[clip下界, log_mel.max()]`，比如 `[-8.0, 0]`。
  - 加 4.0：`[-8.0, 0]` → `[-4.0, 4.0]`（这样范围关于 0 对称）
  - 除 4.0：`[-4.0, 4.0]` → `[-1, 1]`（归一化到 [-1, 1]）
  - **为什么这样做？** 神经网络更容易学习 [-1, 1] 范围的数据（梯度更稳定），而不是原始的 log 能量 [-80, 0] dB。

**验证**：如果你的静音音频（能量都很低），log_mel 会全是 -infinity（被 max 截成 -8.0），那么：
```
(-8.0 + 4.0) / 4.0 = -4.0 / 4.0 = -1.0  ✓
```
最大能量的 mel bin：
```
(0 + 4.0) / 4.0 = 1.0  ✓
```



In [ ]:
# 演示：验证帧数公式
sr = 16000
n_seconds = 30
hop = 160
n_samples = sr * n_seconds
n_frames = n_samples // hop
print(f"30 s @ 16 kHz = {n_samples} 采样点")
print(f"hop={hop} → 帧数 = {n_samples} / {hop} = {n_frames}")
print(f"Whisper 输入 shape: (80, {n_frames})")


### 为什么 Aurora 的 mel_spectrogram 返回 3001 帧而不是 3000 帧？

你可能发现了：30 秒音频理论上应该是 3000 帧，但 Aurora（和官方 Whisper）实际返回 **3001** 帧。为什么多出来 1 帧？

**原因：center=True 的反射填充（reflection padding）**

做 STFT 时，边界处理有个问题：

```
采样点 [S0, S1, S2, ..., S_479999]
         ↑ 第一个采样点处无法开窗（没有"前面"的数据）
```

如果第一帧窗口从 S0 开始，窗口大小 400，那窗口是 [?, ?, ..., ?, S0, S1, ..., S99]——前面 300 个点缺失。

**center=True** 的做法是：在开头和结尾各补充数据，这样能从 S0 的中心处开一个完整窗口。具体补充什么？**反射填充（reflect）**：

```
原始：                   [S0, S1, S2, ..., S_479999]
补充前面 200 个点：      [S200, S199, ..., S1,  S0, S1, S2, ..., S_479999]
                          ↑ 反向镜像，如同音频从边界"反弹回来"

补充后面 200 个点：      [S200, ..., S0, ..., S_479999, S_479998, S_479997, ..., S_479800]
                                                           ↑ 同样反向镜像
```

加了这 200+200=400 个填充点后，总共 400+480000+400=480800 个点。再用 hop=160 采样：

```
480800 / 160 = 3005 帧
```

但官方代码的实现（包括 Aurora）发现了个小"技巧"：不如干脆先补成足够能整除的长度，再额外补 1 帧用不到的数据。结果是 **3001** 帧。这多出来的 1 帧（第 3001 帧）实际上是填充数据的产物，没有真实音频信息，所以练习中你需要**截断到 3000**。

**核心代码逻辑**：
```python
mel_raw = mel_spectrogram(wav, center=True)  # 返回 (3001, 80)
mel_3000 = mel_raw[:3000]  # 截掉末帧 → (3000, 80)
```

这就是为什么 `whisper_preprocess` 的步骤 3 要"pad 或 truncate 到 3000 帧"。



## 2. 编码器：Conv1D × 2 + Transformer → 1500 步上下文向量

Whisper 编码器对 `(80, 3000)` 的 log-Mel 做两层 **Conv1D 降采样**，再送入 Transformer。

```
Conv1d(80 → d, kernel=3, padding=1, stride=1)   # 3000 → 3000
Conv1d(d  → d, kernel=3, padding=1, stride=2)   # 3000 → 1500
```

两层卷积总降采样为 **2 倍**：stride=1 保持 3000 帧，stride=2 压缩至 1500 帧。

### Conv1d 输出长度公式的几何解释

为什么会这样？让我们从**滑动窗口**的角度理解 Conv1d 的 stride 参数。

**基础概念**：Conv1d 的 kernel 就像一个小窗口，沿着序列滑动，每滑一步计算一次输出。

```
输入序列:  [x0, x1, x2, x3, x4, x5, x6, x7, x8, ...]
            ↑
           kernel_size=3 的窗口: [x0, x1, x2]
           → 输出 y0 = 卷积([x0, x1, x2], kernel)
           
            如果 stride=1 (窗口每次向前移 1 个位置)：
            [x0, x1, x2] → y0
               [x1, x2, x3] → y1
                  [x2, x3, x4] → y2
                  ...
            
            如果 stride=2 (窗口每次向前移 2 个位置)：
            [x0, x1, x2] → y0
                  [x2, x3, x4] → y1
                        [x4, x5, x6] → y2
                  ...
```

**stride=2 的作用**：窗口每次跳过 2 个位置，所以输出个数大约是输入的一半——这就是"降采样"。

**公式**：
```
n_out = (n_in + 2*padding - kernel_size) // stride + 1
```

让我们用数字验证 Whisper 的设置（kernel=3, padding=1, stride=1 或 2）：

```
Conv1 (stride=1, padding=1):
  n_out = (3000 + 2*1 - 3) // 1 + 1 = (3000 - 1) // 1 + 1 = 3000 ✓

Conv2 (stride=2, padding=1):
  n_out = (3000 + 2*1 - 3) // 2 + 1 = (3000 - 1) // 2 + 1 = 2999 // 2 + 1 = 1499 + 1 = 1500 ✓
```

**为什么要加 padding=1？**

padding 在输入两端各补 1 个零（或其他值），作用是：
- 不加 padding，kernel 应用于边界时输入不足（例如 kernel=3 需要 3 个输入，但边界附近可能不够）。
- 加 padding，即使在边界也能应用 kernel，防止序列长度快速下降。

```
无 padding 的 Conv1d (kernel=3, stride=1):
  n_out = (n_in - kernel) // stride + 1 = (3000 - 3) // 1 + 1 = 2998
  
有 padding=1 的 Conv1d (kernel=3, stride=1)：
  n_out = (3000 + 2*1 - 3) // 1 + 1 = 3000  ← 长度保持不变
```

**Whisper 的设计选择**：
- 层 1 用 stride=1：保持 3000 帧，只是扩大感受野（receptive field）。
- 层 2 用 stride=2：在有信息的基础上，直接压缩到 1500 帧。
- 为什么不两层都用 stride=2？那样会得到 3000 → 750，压缩太多信息。

注意：如果两层卷积都用 stride=2，会得到 3000 → 1500 → 750——那不是 Whisper。Whisper 官方代码（openai/whisper）只有第二层用 stride=2，且从 tiny 到 large **所有尺寸**的编码器都输出 **1500** 步的上下文序列 `(1500, d_model)`。

```
编码器输出 shape: (B, 1500, d_model)
  d_model: tiny=384, base=512, small=768, medium=1024, large=1280
```

每个 Transformer 层包含：**自注意力（self-attn）→ 前馈网络（FFN）**，编码器不用 causal mask（每帧可以看全局声学上下文）。


In [ ]:
# 演示：用 PyTorch 手写一个极简 Whisper 编码器片段（只展示 Conv 降采样）
import torch.nn as nn
import torch.nn.functional as F

d_model = 512  # Whisper-base
conv1 = nn.Conv1d(80, d_model, kernel_size=3, padding=1, stride=1)
conv2 = nn.Conv1d(d_model, d_model, kernel_size=3, padding=1, stride=2)

dummy_mel = torch.zeros(1, 80, 3000)   # (B, n_mels, T)
x1 = F.gelu(conv1(dummy_mel))          # (1, 512, 3000)
x2 = F.gelu(conv2(x1))                 # (1, 512, 1500)
print(f"输入: {tuple(dummy_mel.shape)} → Conv1 → {tuple(x1.shape)} → Conv2 → {tuple(x2.shape)}")
print(f"编码器将继续接 {x2.shape[-1]} 步的 Transformer 层")


## 3. 解码器：自回归（autoregressive，AR） Transformer + 交叉注意力 → BPE token

解码器是标准的**自回归 causal Transformer**，与 GPT 结构类似，但多了一层 **交叉注意力（cross-attention）**，其 Key/Value 来自编码器输出。

```
每个解码器层的顺序：
  1. 自注意力（causal mask，只看已生成的 token）
  2. 交叉注意力（Q 来自解码器，K/V 来自编码器 1500 步）
  3. FFN
```

### Causal Mask：为什么解码器看不到"未来"？

想象你在做实时直播字幕：

```
实时音频流：[你好] [世界] [，] [很] [高] [兴] ...
         ↑ 正在处理这个词

你的解码器应该：
  ✅ 参考前面已说的词：[你好]（帮助预测当前词）
  ❌ 不能"偷看"后面的词：[世界]、[，]、[很]、... （这会导致过拟合、泄露信息）
```

在 Transformer 自注意力中，**causal mask** 通过设置一个"遮挡矩阵"实现这一点：

```
解码器自注意力权重矩阵（未做 mask）：
        T0  T1  T2  T3  T4
    T0 [1   2   3   4   5]   第 0 步能看所有步
    T1 [6   7   8   9  10]   第 1 步也能看所有步（包括未来的 T2, T3, T4）
    T2 [11 12  13  14  15]   第 2 步也能看所有步...
    T3 [16 17  18  19  20]
    T4 [21 22  23  24  25]

应用 causal mask 后：
        T0  T1  T2  T3  T4
    T0 [1  -∞  -∞  -∞  -∞]   第 0 步只看自己
    T1 [6   7  -∞  -∞  -∞]   第 1 步看 T0 和 T1
    T2 [11 12  13  -∞  -∞]   第 2 步看 T0, T1, T2
    T3 [16 17  18  19  -∞]   第 3 步看 T0, T1, T2, T3
    T4 [21 22  23  24  25]   第 4 步看所有

经过 softmax（-∞ → 0），权重自动"归零"未来的步。
```

**编码器 vs 解码器**：
- **编码器的自注意力**：没有 causal mask，每一帧可以"看"全部 3000 帧（双向上下文）。原因是编码器处理的是**完整的音频**，信息对称。
- **解码器的自注意力**：有 causal mask，第 i 个 token 只能看前 i-1 个 token。原因是解码是**逐步生成**的，不能违反时间顺序。
- **解码器的交叉注意力**：没有 causal mask（对编码器而言），每个解码步都能看编码器的全部 1500 步。理由：编码器已经固定了，没有"未来"的概念，只有"全局"。

词表（vocabulary）大小：**51865**（多语言模型），使用 tiktoken BPE。特殊 token：`<|startoftranscript|>`、语言标签 `<|zh|>`、任务标签 `<|transcribe|>`、`<|notimestamps|>` 或时间戳 token（`<|0.00|>` 到 `<|30.00|>`）。

解码过程：

```
[SOT] → [语言] → [任务] → [内容 token_1] → [token_2] → ... → [EOT]
```

每步解码通过交叉注意力"看"整段 1500 步编码器输出，决定当前发什么音节。

### 追问：交叉注意力为什么比其他方案更好？

上面讲了交叉注意力"长什么样"（Q 来自解码器，K/V 来自编码器），也讲了编码器为什么能看全部 1500 步。但还有一个更根本的问题：**为什么要这样设计？跟其他"看起来也能 work"的方案比，它好在哪？**

#### 替代方案 1：把编码器和解码器的 token "拼在一起"做纯自注意力

既然 Transformer 的自注意力这么好用，为什么不干脆把编码器的 1500 个向量和解码器已经生成的 token 拼成一个长序列，整体做一次自注意力，而不要单独搞一个"交叉注意力"模块？

问题出在**计算量**和**复用效率**上。解码是**逐 token 自回归**的——生成第 1 个字、第 2 个字……最多可能生成 448 个 token。

- **如果拼在一起做自注意力**：解码器每生成一个新 token，理论上都要把"编码器 1500 步 + 目前已生成的所有 token"重新拼一遍、重新算一次自注意力。序列长度 = 1500 + T_dec，计算量 ∝ (1500 + T_dec)²——而且**编码器那 1500 步的信息从头到尾都没变过**，每次都重新参与计算等于白白浪费。
- **交叉注意力的做法**：编码器只需要跑**一次**，算出 1500 个 K、V 向量后就**缓存**起来；后面解码器生成第 2、3、...、448 个 token 时，都直接复用这份缓存的 K/V，不用重新计算编码器。每步交叉注意力的计算量只是 `T_dec_当前 × 1500`，而不是 `(1500+T_dec)²`。

打个比方：编码器的 1500 步就像一本你已经读完、做好摘要（K/V）的参考书——你不需要每写一个字就重新读一遍整本书，只需要**反复查阅同一份摘要**就够了。这正是"K/V 来自编码器"这个设计的实际价值：编码器算一次，解码器可以反复"问"很多次。

#### 替代方案 2：只看最相关的 top-k 步，而不是全部 1500 步

另一个直觉是：既然解码一个字通常只需要参考音频里的一小段（比如 100~150 步），那干脆只让模型看这几步，其余步直接忽略，岂不是更省算力？

问题在于：**你怎么提前知道该看哪几步？** 这需要一个额外的"选择机制"来判断"相关"，而这个选择本身恰恰是注意力机制要解决的问题——如果这个选择是硬性的（选中就选中，没选中就完全看不到），一旦选错了关键帧，模型就**永久丢失**了那部分信息，而且"选 top-k"这一步通常不可导，梯度没法穿过去纠正错误的选择。

交叉注意力的 softmax 机制，其实就是一种**"软性的" top-k**：它照样对全部 1500 步都算一遍相似度，但通过 softmax，真正相关的几步会自然获得接近 1 的权重，不相关的步骤权重会被压得接近 0——效果上跟"只看最相关的几步"很像，但整个过程**连续、可导**，模型在训练时能自己学会"该往哪儿看"，不需要人工划定 k 是多少，也不会因为一次性"选错"就彻底丢信息（权重虽小但不是硬性归零，仍留了一点"备用"信息通道）。

**一句话总结**：交叉注意力 = 编码器只算一次、可反复复用（省计算） + softmax 是可训练的"软性选择"（不用人工划定看哪几步、也不会因为选错而彻底丢信息）。这两点结合起来，比"拼接自注意力"更省算力，比"手工挑 top-k"更灵活可靠。

### 追问：为什么要除以 sqrt(d_k)，而不是 d_k 或别的数？

下面的代码里有一行看起来不起眼的除法：

```python
attn_weight = (Q @ K.transpose(-1,-2)) / (head_dim ** 0.5)
```

`head_dim ** 0.5` 就是 $\sqrt{d_k}$（$d_k$ 是每个注意力头的维度，Whisper-base 是 512/8=64）。这一步在 Transformer 论文（*Attention Is All You Need*, 2017）里叫 **scaled dot-product attention**——标题里的 "scaled" 说的就是这一除法。为什么必须要除？为什么偏偏是平方根，不是 $d_k$ 本身，也不是随便一个常数？我们从头推一遍。

#### 第一步：点积的"方差"是怎么随维度增长的？

Q 和 K 都是长度为 $d_k$ 的向量，点积就是把它们对应位置的分量两两相乘再求和：

```
Q·K = q_1·k_1 + q_2·k_2 + ... + q_{d_k}·k_{d_k}
```

回忆一下 L27-L28 学过的方差（variance）：如果 $q_i$ 和 $k_i$ 都是"均值为 0、方差为 1"的独立随机数（这跟神经网络里常见的参数初始化很接近），那么每一项乘积 $q_i k_i$ 的均值是 0，方差约等于 1。点积是 $d_k$ 个这样的项加起来——**独立随机变量相加，方差也跟着相加**：

```
Var(Q·K) ≈ d_k × 1 = d_k
```

也就是说：**维度 $d_k$ 越大，点积的"摆动幅度"（标准差 = $\sqrt{d_k}$）就越大**。$d_k=64$ 时，点积的标准差大约是 8；$d_k=512$ 时，标准差涨到了约 22.6。

#### 第二步：点积摆动幅度太大，会让 softmax "卡死"

softmax 会把一组数字变成一组加起来等于 1 的权重，但如果输入的数字本身摆动幅度很大（比如有的 60，有的 -50），softmax 会把几乎全部权重都给最大的那个，其余全部逼近 0——这叫**饱和（saturation）**。

饱和为什么是坏事？往前翻到 L24 的链式法则：反向传播要通过 softmax 把梯度传回去，而 softmax 的梯度里有一项形如 $p_i(1-p_i)$（$p_i$ 是某个权重）。如果 $p_i$ 已经非常接近 1 或 0，那 $p_i(1-p_i)$ 就非常接近 0——**梯度几乎传不回去了**，模型学不动。维度越大、不缩放的点积摆动越剧烈，这个问题就越严重。

#### 第三步：为什么除以 $\sqrt{d_k}$ 正好能"解决"这个问题？

我们希望缩放之后，不管 $d_k$ 多大，点积的标准差都能稳定在 1 左右（不大不小，softmax 输入刚好在一个"温和"的范围）：

```
Var(Q·K / c) = Var(Q·K) / c²  ≈  d_k / c²
```

要让这个结果等于 1，只需要 $c^2 = d_k$，也就是 $c = \sqrt{d_k}$。**这就是为什么是平方根，而不是 $d_k$ 本身**——如果除以 $d_k$（而不是 $\sqrt{d_k}$），缩放后的方差会变成 $d_k / d_k^2 = 1/d_k$，维度越大反而缩得越小，点积全部挤在 0 附近，注意力权重会变得几乎"一视同仁"（softmax 输出趋于均匀分布），同样学不出"哪些步更重要"。所以 $\sqrt{d_k}$ 是唯一能让缩放后方差稳定在 1（不随 $d_k$ 变化）的选择——不多不少，正好抵消掉方差里那个 $d_k$ 的增长。

下面用代码验证一下这三步的推导，并看看"不缩放/缩放过头会发生什么"。

In [ ]:
# 演示：d_k 越大，点积方差越大；除以 sqrt(d_k) 能把方差稳定在 1
import numpy as np

np.random.seed(0)
print("第一步验证：Var(Q·K) ≈ d_k")
print("-" * 60)
for d_k in [16, 64, 256, 1024]:
    dots = np.array([np.random.randn(d_k) @ np.random.randn(d_k) for _ in range(3000)])
    print(f"d_k={d_k:5d}  实测 Var(Q·K)={dots.var():8.1f}  理论 ≈ d_k={d_k:5d}  实测 std={dots.std():.2f}  理论 sqrt(d_k)={d_k**0.5:.2f}")

print()
print("第二步验证：不缩放 vs 除以 sqrt(d_k) vs 除以 d_k —— softmax 是否'饱和'")
print("-" * 60)

def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

d_k = 512  # 故意取一个较大的维度，让效果更明显（真实 head_dim 通常是 64）
q = np.random.randn(d_k)
keys = np.random.randn(6, d_k)  # 假设当前解码步要和 6 个编码步比较相似度

raw_scores = keys @ q                       # 不缩放
scaled_scores = raw_scores / np.sqrt(d_k)   # 除以 sqrt(d_k)：Whisper/Transformer 的做法
over_scaled = raw_scores / d_k              # 错误示范：除以 d_k 本身

print("原始点积 raw_scores       :", np.round(raw_scores, 1))
print("除以 sqrt(d_k) 之后        :", np.round(scaled_scores, 2))
print("除以 d_k 之后（过度缩放）  :", np.round(over_scaled, 3))

print("\nsoftmax(不缩放)       :", np.round(softmax(raw_scores), 3), " ← 几乎只有一个权重≈1，其余≈0（饱和，梯度消失）")
print("softmax(除以sqrt(d_k)):", np.round(softmax(scaled_scores), 3), " ← 有主次之分，但不极端")
print("softmax(除以d_k)      :", np.round(softmax(over_scaled), 3), " ← 权重几乎平均，分不出主次（过度缩放）")


In [ ]:
# 演示：交叉注意力的 shape 关系
B, T_enc, T_dec = 1, 1500, 20  # 编码器 1500 步，当前解码到第 20 个 token
d_model, n_heads = 512, 8
head_dim = d_model // n_heads

Q = torch.randn(B, T_dec, d_model)    # 来自解码器当前状态 (1, 20, 512)
K = torch.randn(B, T_enc, d_model)    # 来自编码器输出 (1, 1500, 512)
V = torch.randn(B, T_enc, d_model)    # 来自编码器输出 (1, 1500, 512)

print("=" * 60)
print("交叉注意力的维度推导")
print("=" * 60)
print(f"Q shape (来自解码器): {tuple(Q.shape)}")
print(f"K shape (来自编码器): {tuple(K.shape)}")
print(f"V shape (来自编码器): {tuple(V.shape)}")

# 步骤1：K转置
K_T = K.transpose(-1, -2)
print(f"\nK.transpose(-1, -2): {tuple(K.shape)} → {tuple(K_T.shape)}")
print(f"  （把 T_enc=1500 和 d_model=512 交换位置）")

# 步骤2：Q @ K^T
print(f"\nQ @ K.T 矩阵乘法:")
print(f"  Q:    {tuple(Q.shape)}")
print(f"  K.T:  {tuple(K_T.shape)}")
print(f"  忽略 batch 维，看后两维：")
print(f"    (T_dec, d_model) × (d_model, T_enc)")
print(f"    (20, 512) × (512, 1500)")
print(f"    按矩阵乘法规则 (m,n) × (n,p) = (m,p):")
print(f"    (20, 512) × (512, 1500) = (20, 1500)")

attn_weight = (Q @ K.transpose(-1,-2)) / (head_dim ** 0.5)
print(f"\n注意力权重 shape: {tuple(attn_weight.shape)}")
print(f"  含义：(batch, T_dec, T_enc) = (1, 20, 1500)")
print(f"  第 i 行、第 j 列 = 解码第 i 步对编码第 j 步的原始相似度")

# 步骤3：softmax
attn_weight_norm = torch.softmax(attn_weight, dim=-1)
print(f"\nSoftmax 后的权重: {tuple(attn_weight_norm.shape)}")
print(f"  每一行的 1500 个数字加起来 = 1.0（概率分布）")
print(f"  例如，第 0 行的前 10 个值之和 = {attn_weight_norm[0, 0, :10].sum():.4f}")

# 步骤4：加权求和
out = attn_weight_norm @ V
print(f"\n注意力输出 = 权重 @ V:")
print(f"  attn_weight: {tuple(attn_weight_norm.shape)}")
print(f"  V:           {tuple(V.shape)}")
print(f"  (1, 20, 1500) × (1, 1500, 512) = (1, 20, 512)")
print(f"  out shape:   {tuple(out.shape)}")
print(f"\n解释：20 个解码步中的每一步都“看到”了编码器 1500 步的加权平均。")
print(f"      权重由该步的 Q 和所有编码 K 的相似度决定。")
print("=" * 60)

# 呼应上面 markdown 的推导：为什么这里除的是 sqrt(head_dim) 而不是别的数
# 注意：上面 attn_weight 是直接用完整的 d_model=512 维向量做点积（为了突出 shape 推导，
# 没有真的先按"头"切开）。这里单独切出第 1 个头实际用到的 head_dim=64 维，跟 markdown 里的推导对齐。
print("\n为什么除以 sqrt(head_dim)？用“单头”切片验证一下：")
Q_head = Q[..., :head_dim]           # 模拟切出第 1 个头的 64 维
K_head = K[..., :head_dim]
raw_scores_head = Q_head @ K_head.transpose(-1, -2)
print(f"  head_dim = {head_dim}  → 理论 sqrt(head_dim) = {head_dim ** 0.5:.2f}")
print(f"  未缩放 raw_scores 的标准差: {raw_scores_head.std():.2f}  (理论上 ≈ sqrt(head_dim) = {head_dim ** 0.5:.2f})")
print(f"  除以 sqrt(head_dim) 之后的标准差: {(raw_scores_head / (head_dim ** 0.5)).std():.2f}  (应该 ≈ 1)")
print("  → 缩放把标准差从 sqrt(head_dim) 拉回到 ≈1，不管 head_dim 多大，softmax 输入的“陡峭程度”都差不多。")
print("  （上面完整计算里的 attn_weight 用的是完整 512 维点积，是本课简化版的 shape 演示；")
print("   真实的多头注意力会先把 Q/K/V 切成 n_heads 份、每份 head_dim 维，再各自做这一步缩放。）")


### 3D 张量的矩阵乘法和 transpose 详解

你在高中学的矩阵乘法都是 2D 的：`(m, n) × (n, p) = (m, p)`。但深度学习里经常有 3D 甚至更高维的张量，怎样才能理解 `Q @ K.transpose(-1, -2)` 这样的操作？

**关键原则：PyTorch 在"批处理"维度上不动，只对后面的维度做矩阵乘法。**

**例子**：假设有 2 个 batch，每个 batch 内有一个 (3, 4) 的矩阵和一个 (4, 5) 的矩阵，怎样相乘？

```python
A = torch.randn(2, 3, 4)  # (batch=2, rows=3, cols=4)
B = torch.randn(2, 4, 5)  # (batch=2, rows=4, cols=5)

C = A @ B  # 结果是什么？
```

**直觉**：把 batch 这一维"忽略"，只看后两个维度：
- batch 0：(3, 4) × (4, 5) = (3, 5)
- batch 1：(3, 4) × (4, 5) = (3, 5)
- 所以 `C.shape = (2, 3, 5)`

**验证**：`C[0] = A[0] @ B[0]` 和 `C[1] = A[1] @ B[1]`。

---

**应用到交叉注意力**：在 Whisper 解码器里，我们有：
- Q（Query）来自解码器：shape `(B, T_dec, d_model) = (1, 20, 512)`
- K（Key）来自编码器：shape `(B, T_enc, d_model) = (1, 1500, 512)`
- 我们想算注意力权重：`Q @ K^T`

但 K 是 `(1, 1500, 512)`，我们需要它变成 `(1, 512, 1500)` 才能和 Q 相乘。怎么转？

**Transpose 的含义**：

```python
K = torch.randn(1, 1500, 512)  # (batch, T_enc, d_model)
K.transpose(-1, -2)            # (batch, d_model, T_enc) = (1, 512, 1500)
```

`transpose(-1, -2)` 的含义是"交换倒数第二和倒数第一个维度"。对于 3D 张量 (B, T, D)，就是交换 T 和 D，得到 (B, D, T)。

**现在可以相乘了**：
```
Q:              (1, 20, 512)
K.T:            (1, 512, 1500)

Q @ K.T = ?
```

按照"忽略 batch"的原则：
- batch 维不动：(20, 512) × (512, 1500) = (20, 1500)
- 所以 `(Q @ K.T).shape = (1, 20, 1500)`

这就是**注意力权重矩阵**：每一行代表一个解码步，每一列代表一个编码步，矩阵元素 [i, j] 表示"解码第 i 步对编码第 j 步的关注度"。

**为什么用 transpose(-1, -2) 而不是 transpose(1, 2)？**

`transpose(-1, -2)` 在面对不同维度的张量时更通用：
- 3D：`(B, T, D).transpose(-1, -2)` → `(B, D, T)` ✓
- 4D：`(B, H, T, D).transpose(-1, -2)` → `(B, H, D, T)` ✓（多头注意力）
- 用 `transpose(1, 2)` 在 4D 时就错了



## 4. 多任务头：一个解码器，四种输出

Whisper 的独特之处不仅是 encoder-decoder 架构，而是**同一个模型用特殊 token 条件化实现四个不同任务**：

| 任务 | 头输出 | 条件 token |
|------|--------|-----------|
| 语音识别（transcribe） | BPE 文字序列 | `[SOT][lang][transcribe][notimestamps]` |
| 语音翻译（translate） | 英文 BPE 序列 | `[SOT][lang][translate][notimestamps]` |
| 语言识别（language-ID） | lang token 概率 | `[SOT]` → 取下一 token 分布 |
| 带时间戳识别 | BPE + timestamp token 交错 | `[SOT][lang][transcribe]`（无 notimestamps） |

### 前缀 Token 如何"指挥"模型做不同的任务？

**核心思想**：Whisper 不是靠"两个不同的模型"或"不同的架构分支"来区分任务，而是靠**一个共享的 embedding 和参数**，让前缀 token 的含义在**训练阶段**被编码进来。

**类比**：想象你有一个智能助手（模型），有一个"指令卡"系统：

```
情景 A：我说"天气怎样？"
  → 你问天气API → 回答天气

情景 B：我说"翻译成英文"，然后给出中文句子
  → 你知道要翻译 → 做翻译任务

模型也是这样：
  • 看到 prefix = [SOT, LANG_ZH, TRANSCRIBE, NO_TIMESTAMPS]
    → "啊，这是一个'转写'任务，我要把中文语音转成中文文字"
  
  • 看到 prefix = [SOT, LANG_ZH, TRANSLATE, NO_TIMESTAMPS]
    → "啊，这是一个'翻译'任务，我要把中文语音转成英文"
```

**训练过程中发生了什么**：

Whisper 在**超过 680,000 小时**的多语言音频上训练，数据混合了各种任务的标注：

```
数据示例 1（转写）：
  输入 audio: [中文语音...]
  前缀:      [SOT, LANG_ZH, TRANSCRIBE, NO_TIMESTAMPS]
  目标 tokens: [你, 好, 世, 界]
  损失函数:   CrossEntropy(预测, 目标)

数据示例 2（翻译）：
  输入 audio: [同一段中文语音...]
  前缀:      [SOT, LANG_ZH, TRANSLATE, NO_TIMESTAMPS]
  目标 tokens: [hello, world]  # 英文翻译
  损失函数:   CrossEntropy(预测, 目标)

数据示例 3（语言识别）：
  输入 audio: [中文语音...]
  前缀:      [SOT]
  目标 token: LANG_ZH  # 只预测语言，不生成内容
```

经过大量训练，模型学会了：
- **TRANSCRIBE token** 的 embedding 激活了"识别并转写"的神经回路
- **TRANSLATE token** 的 embedding 激活了"识别并翻译成英文"的神经回路
- **LANG_ZH token** 告诉模型"输入是中文"（改变 softmax 概率分布，让中文字符概率更高）

**关键：这一切都在 Transformer 的自注意力和交叉注意力中发生**。前缀 token 通过注意力机制"广播"它的信息到后续所有 token：

```
解码器自注意力权重（简化示意）：
        SOT  LANG  TRANSCRIBE  NO_TS  内容1  内容2
    SOT [ 1    0      0         0      0      0 ]  （SOT 自己关注自己）
    LANG[ 0.3  1      0.2       0.1    0      0 ]  （语言token关注SOT、自己、task）
    TRANSCRIBE[ 0.2 0.3  1     0.2    0.1    0 ]  （任务token关注前面的条件）
    NO_TS[ 0.1  0.2  0.3  1      0.2    0.1 ]   （格式token关注所有前缀）
    内容1[ 0.1  0.15 0.25 0.2  1       0.2 ]   （内容词看到整个前缀）
    内容2[ 0.1  0.1  0.2  0.15 0.3    1   ]   （后续内容词看整个历史）
```

注意力权重告诉我们：**后续 token 生成时都"加权平均"了前缀 token 的信息**。比如"内容 1"的 embedding 会被"TRANSCRIBE"的 embedding 影响 25%，被"LANG_ZH"的 embedding 影响 15%。这样，前缀中"翻译"vs"转写"的区别就能指导内容 token 的生成。

---

**关键设计**：所有任务共享同一个 **输出投影层**（`Linear(d_model, vocab_size)`），任务区分完全靠前缀 token 序列。也就是说，训练时只要把四类数据混在一起，推理时在 `[SOT]` 后拼接不同特殊 token 就能切换任务。

```
解码时 token 前缀（以 transcribe 为例）：
  [SOT=50258] [zh=50260] [transcribe=50359] [notimestamps=50363] → content tokens...
```

vocab_size = **51865**（50257 BPE + 1608 特殊 token），其中 timestamp token 占 1501 个（`<|0.00|>` 到 `<|30.00|>`，步长 0.02 s，含两端）。


In [ ]:
# 演示：多任务头 token 条件化
# 特殊 token ID（来自 openai/whisper 多语言 tokenizer）
SOT         = 50258   # start of transcript
LANG_ZH     = 50260   # 中文
TRANSCRIBE  = 50359   # 转录任务
TRANSLATE   = 50358   # 翻译任务
NO_TIMESTAMPS = 50363 # 不输出时间戳（50362 是 <|nospeech|>）

# 三种任务的前缀序列
task_prefixes = {
    "识别（中文）": [SOT, LANG_ZH, TRANSCRIBE, NO_TIMESTAMPS],
    "翻译→英文":   [SOT, LANG_ZH, TRANSLATE, NO_TIMESTAMPS],
    "语言识别":    [SOT],            # 只看下一个 token 的分布
}
for task, prefix in task_prefixes.items():
    print(f"{task}: prefix tokens = {prefix}")

# 输出投影层 shape
import torch.nn as nn
vocab_size = 51865
d_model = 512  # Whisper-base
output_proj = nn.Linear(d_model, vocab_size, bias=False)
print(f"\n共享输出投影层: Linear({d_model}, {vocab_size}) = {sum(p.numel() for p in output_proj.parameters()):,} 参数")
print("所有任务共享同一 output_proj，切换任务只需改前缀 token。")

## 5. ✏️ 实现 `whisper_preprocess(wav, sr=16000)`

**签名**：
```python
def whisper_preprocess(wav: np.ndarray, sr: int = 16000) -> torch.Tensor:
    # 返回: (1, 80, 3000) float32 Tensor（batch_size=1，通道=1被省略）
```

**4步实现路线**：

| 步骤 | 操作 | 参数 |
|---|---|---|
| 1 | `mel_spectrogram(wav, sr, ...)` | n_fft=400, hop=160, n_mels=80 |
| 2 | `power_to_db(...)` | 转 dB（log-Mel），top_db=80 |
| 3 | pad/truncate → 固定3000帧 | `mel.T` shape=(T, 80)，T 可能≠3000（30 s 恰好是 3001） |
| 4 | `torch.tensor(...).float().unsqueeze(0)` | (80, 3000) → (1, 80, 3000) |

**验收标准**：
- `whisper_preprocess(np.zeros(16000*30)).shape == (1, 80, 3000)`
- `dtype == torch.float32`
- 30秒以内音频 pad 到 3000 帧，超出截断

### 补充：编码器为什么输出 1500 步向量？

你可能问：为什么卷积层把 3000 帧降到 1500 帧后，Transformer 层就不再改变序列长度了？

**答案**：Transformer 的自注意力和前馈网络都是**逐位置（position-wise）**的操作，不改变序列长度。

```
输入：(B, T, d_model) = (1, 1500, 512)
  ↓ 自注意力（每个位置都"看"全部 T=1500 步，但输出还是 1500 个向量）
  ↓ 前馈网络（每个位置独立处理）
输出：(B, T, d_model) = (1, 1500, 512)
```

**为什么卷积要降采样，而 Transformer 不降采样？**

- **卷积**做的是"特征提取"和"降噪"：3000 帧的原始 mel 谱图太冗余了，相邻帧之间差异很小。卷积层通过 stride=2 把冗余降掉。
- **Transformer**做的是"全局关联"和"上下文聚合"：1500 个"稀疏"的特征帧足以覆盖 30 秒的音频（平均每个 Transformer 步对应 20 ms 的音频）。自注意力让每个步看到全部 1500 步的上下文，足以推理出"这是什么音素"。

**每个 Transformer 输出向量代表什么？**

```
Encoder 输出: (1, 1500, 512) — 1500 个向量
      ↓
      第 i 个向量（第 i 步的输出）代表：
      • 时间对齐：大约对应音频的 i × 20ms 处（因为 hop=160 是 10ms，
                                    两层卷积相当于再×2，共 20ms）
      • 语义含义：这一时刻的音频"声学特征"（pitch、formants、静音等）
                和"全局上下文"（前后音素信息）的混合表示
      
      这 512 维的向量被称为 "context vector" 或 "acoustic embedding"
```

**Decoder 的工作**：

解码器读这 1500 个向量，通过交叉注意力"问"每一步"这个 token 对应哪段音频"。比如：
- 当解码"你"这个字时，交叉注意力可能会给编码器的第 100~150 步赋予高权重
- 当解码"好"这个字时，可能给第 200~250 步赋予高权重
- ...以此类推

所以 Transformer 保持 1500 步，目的是**留出足够的时间分辨率**，让后续的交叉注意力能"精确定位"每个 token 对应的音频段。



In [ ]:
def whisper_preprocess(wav: np.ndarray, sr: int = 16000) -> torch.Tensor:
    """
    将原始波形转换为 Whisper 所需的 log-Mel 特征张量。

    参数
    ----
    wav : np.ndarray, shape (T,)
        单声道 PCM 波形，采样率应为 sr。
    sr : int
        采样率，Whisper 默认 16000 Hz。

    返回
    ----
    torch.Tensor, shape (1, 80, 3000), dtype=float32
    """
    # ✏️ TODO: 步骤1 — 调用 aurora mel_spectrogram，参数 n_fft=400, hop_length=160, n_mels=80
    #          注意返回 shape 是 (n_frames, 80)，需要 .T 转置

    # ✏️ TODO: 步骤2 — log 归一化（Whisper 风格）
    #          log_mel = log10(max(mel, 1e-10))
    #          log_mel = max(log_mel, log_mel.max() - 8.0)
    #          log_mel = (log_mel + 4.0) / 4.0

    # ✏️ TODO: 步骤3 — pad 或 truncate 到 3000 帧（axis=1）

    # ✏️ TODO: 步骤4 — 转 torch.float32 tensor，unsqueeze(0)
    raise NotImplementedError("whisper_preprocess: 按步骤1-4实现本函数")


In [ ]:
try:
    # 检查：30 秒静音
    feat = whisper_preprocess(np.zeros(16000 * 30))
    assert feat.shape == (1, 80, 3000), f"shape 错误: {feat.shape}"
    assert feat.dtype == torch.float32, f"dtype 错误: {feat.dtype}"
    print(f"✅ shape={tuple(feat.shape)}, dtype={feat.dtype}")
    print(f"✅ 值域: [{feat.min():.3f}, {feat.max():.3f}]（纯静音时为常数 -1.5）")

    # 检查：短音频自动补零
    feat_short = whisper_preprocess(np.zeros(16000 * 5))  # 5 秒
    assert feat_short.shape == (1, 80, 3000), f"短音频补零失败: {feat_short.shape}"
    print(f"✅ 5 秒短音频自动补零 → shape={tuple(feat_short.shape)}")

    # 检查：长音频自动截断
    feat_long = whisper_preprocess(np.zeros(16000 * 60))  # 60 秒
    assert feat_long.shape == (1, 80, 3000), f"长音频截断失败: {feat_long.shape}"
    print(f"✅ 60 秒长音频截断 → shape={tuple(feat_long.shape)}")
except (NotImplementedError, TypeError):
    print('⚠️ whisper_preprocess 尚未实现——完成步骤1-4后再运行此检查')


## 6. 参数实验：Aurora vs. Whisper 官方 log-Mel 的差异

对比 `aurora.audio.mel.mel_spectrogram` 与 openai/whisper 官方的 `log_mel_spectrogram` （`whisper/audio.py`），找出以下参数的差异：

| 参数 | Aurora 默认 | Whisper 官方 |
|------|-------------|-------------|
| `n_fft` | 1024 | **400** |
| `hop_length` | `n_fft // 4 = 256` | **160** |
| `n_mels` | 80 | 80（相同）|
| `fmin` | 0.0 | **0.0**（相同）|
| `fmax` | `sr / 2` | **sr / 2**（相同）|
| log 归一化 | `power_to_db`（dB，`10*log10`）| **`log10` + clip(-8) + /4 归一化** |
| 输出 shape | `(n_frames, n_mels)` | **`(n_mels, n_frames)`** |

**预期现象**：
- 用 Aurora 默认参数（`n_fft=1024, hop=256`）时，30 s 音频输出 `(1876, 80)` → 转置后 `(80, 1876)`，  不足 3000 帧，补零后与官方结果的**时间分辨率**不同（频率分辨率也更高）。
- 用 Whisper 参数（`n_fft=400, hop=160`）时，30 s 输出 `(3001, 80)`——center padding 多出 1 帧，截掉末帧后与官方的 3000 帧对齐。
- 归一化方式不同：`power_to_db` 输出 dB 值（如 -80 ~ 0 dB），  而 Whisper 归一化后值域约 `[-1, 1]`。


In [ ]:
# 实验1：Aurora 默认参数 vs Whisper 参数的帧数差异
wav_30s = np.zeros(16000 * 30)

# Aurora 默认 (n_fft=1024, hop=256)
mel_aurora_default = mel_spectrogram(wav_30s, 16000, n_fft=1024, hop_length=256, n_mels=80)
print(f"Aurora 默认参数 → mel shape: {mel_aurora_default.shape}  (帧数={mel_aurora_default.shape[0]})")

# Whisper 参数 (n_fft=400, hop=160)
mel_whisper = mel_spectrogram(wav_30s, 16000, n_fft=400, hop_length=160, n_mels=80)
print(f"Whisper 参数   → mel shape: {mel_whisper.shape}  (帧数={mel_whisper.shape[0]})")

# 实验2：log 归一化对比
mel_raw = mel_spectrogram(wav_30s[:16000], 16000, n_fft=400, hop_length=160, n_mels=80)  # 1 秒

# Aurora power_to_db 风格
db = power_to_db(mel_raw)
print(f"\npower_to_db 值域: [{db.min():.1f}, {db.max():.1f}] dB")

# Whisper 风格
log_mel = np.log10(np.maximum(mel_raw, 1e-10))
log_mel = np.maximum(log_mel, log_mel.max() - 8.0)
log_mel = (log_mel + 4.0) / 4.0
print(f"Whisper 归一化 值域: [{log_mel.min():.3f}, {log_mel.max():.3f}]")
print("\n✅ 结论：喂给官方 Whisper 编码器必须用 n_fft=400, hop=160，并做 Whisper 风格归一化")


## 🎯 未来的回报 (Future Payoff)

今天你亲手拆开的这条 **log-Mel → Encoder → 交叉注意力 Decoder → 多任务头** 数据流，会在后面反复出现——值得现在就把每个 shape 记牢：

- **L71**：解码器如何"一个 token 一个 token"地生成——贪婪 vs beam search，正是作用在本课画出的解码器输出分布上。
- **L72**：LoRA 微调只往 `q_proj`/`v_proj` 注入低秩分支，而这两个投影就活在你刚解剖的注意力层里；不懂架构就不知道该往哪儿插。
- **L75**：交叉注意力热力图可视化——"decoder 第 k 个字在看哪几帧音频"，画的就是本课 shape 关系的落地图。

一句话：Whisper 之后所有的**微调、解码、可视化**，都建立在你今天弄清的这张数据流图上。


## 本课收束

本节实现了 `whisper_preprocess(wav)`，把 30 秒音频整理成 Whisper 需要的 `(1, 80, 3000)` 输入；这一步把 Aurora DSP 和 Whisper 编码器真正连了起来。核心依赖 `aurora.audio.mel.mel_spectrogram`，通过调整 `n_fft=400, hop_length=160` 对齐 Whisper 的声学参数，并加入 Whisper 风格 log 归一化（`log10 → clip → /4`）。下一节（L71）将实现 Whisper 的两种解码策略：贪婪解码（每步取最高概率 token）和宽度为 2 的 beam search，对比两者在玩具语言模型上的序列质量差异。

## ✏️ 闭卷推导检查格 — Whisper 数据流图

**规则：关闭上方所有格，仅凭记忆完成以下推导。**

**题目**：Whisper-small 参数：
- 输入：30秒音频，80-mel，帧移 10ms → 3000 帧
- Encoder：12 层 Transformer，d_model=768，12 头（每头 64 维）
- Decoder：输入 BOS token，自回归生成

画出数据流图并标注每一步的**张量形状**：

| 步骤 | 操作 | 输出形状 |
|------|------|---------|
| 1 | log-mel 输入 | (batch, 80, 3000) |
| 2 | Conv1 (kernel=3, stride=1, out=768) | (batch, ___, ___) |
| 3 | Conv2 (kernel=3, stride=2, out=768) | (batch, ___, ___) |
| 4 | Encoder 输出 | (batch, ___, ___) |
| 5 | Decoder 交叉注意力 K/V 来自 | 步骤 ___ 的输出 |

（在此处填表...）

In [ ]:
# 验证：Whisper 卷积层的输出形状计算（不需要加载模型权重）
import numpy as np

def conv1d_out(L_in, kernel, stride=1, padding=1):
    return (L_in + 2*padding - kernel) // stride + 1

T = 3000   # 3000 mel 帧
after_conv1 = conv1d_out(T, kernel=3, stride=1, padding=1)  # 保持长度
after_conv2 = conv1d_out(after_conv1, kernel=3, stride=2, padding=1)  # 下采样 2x

# Whisper-small encoder 序列长度 = 1500
assert after_conv1 == 3000, f"Conv1 输出 {after_conv1}，期望 3000"
assert after_conv2 == 1500, f"Conv2 输出 {after_conv2}，期望 1500"
print(f"Conv1 输出序列长度: {after_conv1}")
print(f"Conv2 输出序列长度: {after_conv2}")
print(f"✅ Whisper 维度推导正确：(batch, 768, 1500) 进入 Transformer Encoder")

### 补充：解码什么时候停止？T_dec 的真正含义

在前面的交叉注意力演示中，我们设置 `T_dec = 20`，表示解码器已经生成了 20 个 token。但这个"20"是固定的吗？一句话可能只有 10 个 token，也可能有 100 个。那什么时候停止解码？

**答案**：解码器会生成一个特殊的 **"结束 token"（EOT，End-Of-Transcript）**，当生成这个 token 时，解码停止。

```
解码过程（贪婪策略示意）：
  
  Step 0（初始化）：输入前缀 [SOT, LANG_ZH, TRANSCRIBE, NO_TIMESTAMPS]
                   Encoder 输出 1500 个向量（不变）
  
  Step 1：Decoder 自回归生成第 1 个内容 token
         - 前缀 + 已生成内容 = [SOT, LANG_ZH, TRANSCRIBE, NO_TIMESTAMPS]（目前没有内容）
         - 通过自注意力和交叉注意力计算
         - 输出 softmax 分布 (vocab_size=51865,)
         - 取最高概率的 token ID → 比如 "你"(ID=4321)
  
  Step 2：继续生成第 2 个 token
         - 前缀 + 已生成内容 = [..., 你]
         - 再次通过解码器（T_dec 现在 = 5, 包括 4 个前缀 + 1 个已生成内容）
         - 输出分布
         - 取最高概率 token → 比如"好"(ID=5678)
  
  Step 3, 4, ..., N：重复上述过程
  
  Step N：模型输出 EOT(ID=50257)
         → 解码停止，输出 [你, 好, 世, 界]（去掉前缀和 EOT）
```

**T_dec 的实际含义**：

在 Transformer 解码器中，`T_dec` 不是"固定的 20"，而是**当前已生成的序列长度（包括前缀）**。

```
初始：T_dec = len(prefix) = 4  （[SOT, LANG_ZH, TRANSCRIBE, NO_TIMESTAMPS]）
Step 1 后：T_dec = 5  （前缀 + 1 个内容 token）
Step 2 后：T_dec = 6  （前缀 + 2 个内容 token）
...
Step N 后：T_dec = len(prefix) + N
```

**实现细节**：

```python
def whisper_decode_greedy(encoder_output, prefix_tokens, max_tokens=448):
    """
    encoder_output: (1, 1500, 512)  — 编码器输出（固定）
    prefix_tokens: [SOT, LANG, TASK, NO_TIMESTAMPS]  — 前缀（固定）
    max_tokens: 最多生成多少个内容 token（防止无限生成）
    """
    generated = prefix_tokens.copy()  # 初始化
    
    for step in range(max_tokens):
        # 当前已生成序列
        current_seq = torch.tensor(generated).unsqueeze(0)  # (1, T_dec)
        
        # 通过解码器（T_dec = len(generated) 在变化）
        logits = decoder(current_seq, encoder_output, causal_mask=True)
        # logits: (1, T_dec, vocab_size)
        
        # 只取最后一步的输出（最新生成的位置）
        next_token_logits = logits[0, -1, :]  # (vocab_size,)
        
        # 贪婪选择：取最高概率 token
        next_token = next_token_logits.argmax()
        
        if next_token == EOT_TOKEN_ID:
            break
        
        generated.append(next_token)
    
    return generated[len(prefix_tokens):]  # 返回内容部分（去掉前缀）
```

**为什么实现中"只取最后一步的输出"？**

注意这一行：
```python
next_token_logits = logits[0, -1, :]  # 只取最后一步
```

虽然解码器一次输出 T_dec 个位置的 logits，但我们只关心**最新添加的 token 的分布**。原因是：
- 前面的位置（已生成的 token）的 logits 不再需要（因为这些 token 已经确定了）
- 只有"下一个要生成的位置"（第 T_dec 步）需要采样

**总结**：
- `T_dec` 不是固定的，它随着生成而增长：4 → 5 → 6 → ...
- 解码何时停止：当模型输出 EOT token 时
- 为了防止无限循环，通常设置 `max_tokens`（比如 448，对应 30 秒音频的平均 token 密度）



In [ ]:
# ✏️ 本课自评
l70_review = {
    "whisper_3stage_understood":  None,  # 记住Whisper三阶段：log-Mel→Encoder→Decoder→BPE？True/False
    "frame_count_3000":           None,  # 理解30s@16kHz用hop=160→3000帧的计算？True/False
    "cross_attention_understood": None,  # 理解解码器cross-attention：Q来自decoder，K/V来自encoder？True/False
    "whisper_preprocess_impl":    None,  # whisper_preprocess()实现正确，shape/dtype验证通过？True/False
    "whiteboard_passed":          None,  # 白板挑战Whisper数据流图推导通关？True/False
}

unfilled = [k for k, v in l70_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l70_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L70 全部通关！进入 L71：Whisper 解码策略')

---

→ **下一课**　[L71 · Whisper 解码策略](L71_whisper_decoding.ipynb)

> 下节课将学习 **Whisper 解码策略**：贪婪解码与 beam search 从原理到实现。